In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [2]:
#Set display options for better readability
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

In [3]:
# =============================================================================
# 1. LOAD DATA
# =============================================================================
print("STEP 1: Loading Data Files")
print("-" * 80)

# Replace with your actual file paths
customer_file = #redacted
vendor_file = #redacted
sales_file = #redacted

try:
    # Load customer data
    excel_customers = pd.ExcelFile(customer_file)
    print(f"\nCustomer file sheets: {excel_customers.sheet_names}")
    df_customers = pd.read_excel(customer_file, sheet_name=0)
    
    # Load vendor data
    excel_vendors = pd.ExcelFile(vendor_file)
    print(f"Vendor file sheets: {excel_vendors.sheet_names}")
    df_vendors = pd.read_excel(vendor_file, sheet_name=0)
    
    # Load posted sales data
    excel_sales = pd.ExcelFile(sales_file)
    print(f"Posted sales file sheets: {excel_sales.sheet_names}")
    df_sales = pd.read_excel(sales_file, sheet_name=0)
    
    print(f"\n✓ Data loaded successfully!")
    print(f"  - Customers: {df_customers.shape[0]:,} rows × {df_customers.shape[1]} columns")
    print(f"  - Vendors: {df_vendors.shape[0]:,} rows × {df_vendors.shape[1]} columns")
    print(f"  - Posted Sales: {df_sales.shape[0]:,} rows × {df_sales.shape[1]} columns")
    
except FileNotFoundError as e:
    print(f"⚠ Error: {e}")
    print("\n⚠ Please update the file paths above with your actual Excel files")
    print("  Example: customer_file = 'C:/Users/Thomas/Desktop/IFC_Data/customers.xlsx'")

SyntaxError: invalid syntax (3999957104.py, line 8)

In [4]:
# =============================================================================
# 2. INITIAL DATA INSPECTION
# =============================================================================
print("\n" + "=" * 80)
print("STEP 2: Initial Data Inspection")
print("-" * 80)

def inspect_dataframe(df, name):
    """Comprehensive initial inspection of a dataframe"""
    print(f"\n📊 {name}")
    print("-" * 40)
    print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"\nColumn Names & Types:")
    print(df.dtypes)
    print(f"\nFirst 3 rows:")
    print(df.head(3))
    print(f"\nBasic Statistics:")
    print(df.describe(include='all').T)
    print(f"\nMissing Values:")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_df = pd.DataFrame({
        'Missing_Count': missing,
        'Missing_Percentage': missing_pct
    })
    print(missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False))

# Inspect all three dataframes
inspect_dataframe(df_customers, "CUSTOMERS")
inspect_dataframe(df_vendors, "VENDORS")
inspect_dataframe(df_sales, "POSTED SALES")


STEP 2: Initial Data Inspection
--------------------------------------------------------------------------------


NameError: name 'df_customers' is not defined

In [5]:
# =============================================================================
# 3. DATA CLEANING
# =============================================================================
print("\n" + "=" * 80)
print("STEP 3: Data Cleaning")
print("-" * 80)

def clean_dataframe(df, name):
    """Clean and standardize dataframe"""
    df_clean = df.copy()
    
    print(f"\nCleaning {name}...")
    
    # Remove completely empty rows
    initial_rows = len(df_clean)
    df_clean = df_clean.dropna(how='all')
    removed_empty = initial_rows - len(df_clean)
    if removed_empty > 0:
        print(f"  ✓ Removed {removed_empty} completely empty rows")
    
    # Remove duplicate rows
    initial_rows = len(df_clean)
    df_clean = df_clean.drop_duplicates()
    removed_dupes = initial_rows - len(df_clean)
    if removed_dupes > 0:
        print(f"  ✓ Removed {removed_dupes} duplicate rows")
    
    # Clean column names (lowercase, remove spaces)
    df_clean.columns = df_clean.columns.str.lower().str.replace(' ', '_').str.replace('[^a-z0-9_]', '', regex=True)
    print(f"  ✓ Standardized column names")
    
    # Detect and convert date columns
    date_columns = []
    for col in df_clean.columns:
        if df_clean[col].dtype == 'object':
            # Try to convert to datetime
            try:
                if pd.to_datetime(df_clean[col], errors='coerce').notna().sum() / len(df_clean) > 0.5:
                    df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')
                    date_columns.append(col)
            except:
                pass
    
    if date_columns:
        print(f"  ✓ Converted to datetime: {', '.join(date_columns)}")
    
    # Strip whitespace from string columns
    string_cols = df_clean.select_dtypes(include=['object']).columns
    for col in string_cols:
        df_clean[col] = df_clean[col].str.strip() if df_clean[col].dtype == 'object' else df_clean[col]
    
    print(f"  ✓ Cleaned {len(string_cols)} text columns")
    print(f"\n  Final shape: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")
    
    return df_clean

# Clean all three dataframes
df_customers_clean = clean_dataframe(df_customers, "Customers")
df_vendors_clean = clean_dataframe(df_vendors, "Vendors")
df_sales_clean = clean_dataframe(df_sales, "Posted Sales")


STEP 3: Data Cleaning
--------------------------------------------------------------------------------


NameError: name 'df_customers' is not defined

In [7]:
# =============================================================================
# 4. DATA OVERVIEW SUMMARY
# =============================================================================
print("\n" + "=" * 80)
print("STEP 4: Data Overview Summary")
print("-" * 80)

print("\n QUICK SUMMARY")
print("-" * 40)
print(f"Total Customers: {len(df_customers_clean):,}")
print(f"Total Vendors: {len(df_vendors_clean):,}")
print(f"Total Sales Transactions: {len(df_sales_clean):,}")

# Check for date ranges in sales data
date_cols_sales = df_sales_clean.select_dtypes(include=['datetime64']).columns
if len(date_cols_sales) > 0:
    print(f"\nSales Transaction Date Range:")
    for col in date_cols_sales:
        print(f"  {col}: {df_sales_clean[col].min()} to {df_sales_clean[col].max()}")


STEP 4: Data Overview Summary
--------------------------------------------------------------------------------

 QUICK SUMMARY
----------------------------------------


NameError: name 'df_customers_clean' is not defined

In [8]:
# =============================================================================
# 5. CREATE ENTITY LISTS
# =============================================================================
print("\n" + "=" * 80)
print("STEP 5: Creating Entity Lists")
print("-" * 80)

def create_entity_list(df, id_col=None, name_col=None, additional_cols=None):
    """Create a unique list of entities from a dataframe"""
    if id_col and id_col in df.columns:
        cols_to_include = [id_col]
        
        if name_col and name_col in df.columns:
            cols_to_include.append(name_col)
        
        if additional_cols:
            for col in additional_cols:
                if col in df.columns:
                    cols_to_include.append(col)
        
        entities = df[cols_to_include].drop_duplicates()
        return entities.sort_values(by=id_col).reset_index(drop=True)
    return None

customers = create_entity_list(
    df1__customer_clean, 
    id_col='no',
    name_col='name',
    additional_cols=['countryregion_code', 'currency_code', 'vat_registration_no','enterprise_no', 'vat_bus_posting_group', 'gen_bus_posting_group', 'customer_posting_group', 'payment_terms_code', 'balance_lcy', 'balance_due_lcy', 'credit_limit_lcy', 'sales_lcy', 'net_change', 'reminder_terms_code', 'blocked', 'last_date_modified', 'salesperson_code', 'phone_no', 'contact']
)

if customers is not None:
    print(f"\n CUSTOMER LIST")
    print(f"   Total Unique Customers: {len(customers):,}")
    print(f"\nFirst 10 customers:")
    print(customers.head(10))
    print(f"\nLast 10 customers:")
    print(customers.tail(10))
    
    # Save customer list
    customers.to_excel('customer_list.xlsx', index=False)
    print(f"\n✓ Customer list saved to: customer_list.xlsx")


vendors = create_entity_list(
    df2__vendors_clean,  
    id_col='no',
    name_col='name',
    additional_cols=['countryregion_code', 'currency_code', 'vat_registration_no','enterprise_no', 'vat_bus_posting_group', 'gen_bus_posting_group', 'vendor_posting_group', 'balance_lcy', 'balance_due_lcy', 'balance', 'payment_terms_code', 'blocked', 'last_date_modified', 'language_code', 'contact', 'search_name', 'phone_no']
)

if vendors is not None:
    print(f"\n VENDOR LIST")
    print(f"   Total Unique Vendors: {len(vendors):,}")
    print(f"\nFirst 10 vendors:")
    print(vendors.head(10))
    
    # Save vendor list
    vendors.to_excel('vendor_list.xlsx', index=False)
    print(f"\n✓ Vendor list saved to: vendor_list.xlsx")


STEP 5: Creating Entity Lists
--------------------------------------------------------------------------------


NameError: name 'df1__customer_clean' is not defined

In [9]:
# =============================================================================
# 6. COLUMN REFERENCE (For updating code above)
# =============================================================================
print("\n" + "=" * 80)
print("STEP 6: Column Names Reference")
print("-" * 80)

print("\n CUSTOMER COLUMNS:")
for i, col in enumerate(df_customers_clean.columns, 1):
    print(f"  {i}. {col}")

print("\n VENDOR COLUMNS:")
for i, col in enumerate(df_vendors_clean.columns, 1):
    print(f"  {i}. {col}")

print("\n POSTED SALES COLUMNS:")
for i, col in enumerate(df_sales_clean.columns, 1):
    print(f"  {i}. {col}")


STEP 6: Column Names Reference
--------------------------------------------------------------------------------

 CUSTOMER COLUMNS:


NameError: name 'df_customers_clean' is not defined

In [10]:
# =============================================================================
# 7. PAYMENT BEHAVIOR ANALYSIS PREPARATION
# =============================================================================
print("\n" + "=" * 80)
print("STEP 7: Payment Behavior Analysis Preparation")
print("-" * 80)

# Check if sales data can be linked to customers
print("\nLinking Analysis:")

# Look for customer identifier in sales data
customer_link_cols = [col for col in df_sales_clean.columns if 'customer' in col.lower() or 'sell' in col.lower() or 'bill' in col.lower()]
if customer_link_cols:
    print(f" Potential customer link columns in sales data: {customer_link_cols}")
else:
    print(" No obvious customer link column found in sales data")
    print("  Common names: customer_no, sell_to_customer_no, bill_to_customer_no")

# Look for payment-related columns
payment_cols = [col for col in df_sales_clean.columns if any(term in col.lower() for term in ['payment', 'paid', 'due', 'date'])]
if payment_cols:
    print(f"\n Payment-related columns found: {payment_cols}")
else:
    print("\n No obvious payment columns found")

# Look for amount columns
amount_cols = [col for col in df_sales_clean.columns if any(term in col.lower() for term in ['amount', 'value', 'total'])]
if amount_cols:
    print(f"\n Amount columns found: {amount_cols}")


STEP 7: Payment Behavior Analysis Preparation
--------------------------------------------------------------------------------

Linking Analysis:


NameError: name 'df_sales_clean' is not defined

In [11]:
# =============================================================================
# 8. INITIAL PAYMENT BEHAVIOR METRICS
# =============================================================================
print("\n" + "=" * 80)
print("STEP 8: Initial Payment Behavior Metrics")
print("-" * 80)

print("\nSample of Posted Sales Data:")
print(df_sales_clean.head(10))

print("\nSales Data Summary Statistics:")
numeric_cols = df_sales_clean.select_dtypes(include=[np.number]).columns
if len(numeric_cols) > 0:
    print(df_sales_clean[numeric_cols].describe())


STEP 8: Initial Payment Behavior Metrics
--------------------------------------------------------------------------------

Sample of Posted Sales Data:


NameError: name 'df_sales_clean' is not defined

In [13]:
# =============================================================================
# 9. SAVE CLEANED DATA
# =============================================================================
print("\n" + "=" * 80)
print("STEP 9: Saving Cleaned Data")
print("-" * 80)

try:
    df_customers_clean.to_excel('customers_clean.xlsx', index=False)
    df_vendors_clean.to_excel('vendors_clean.xlsx', index=False)
    df_sales_clean.to_excel('sales_clean.xlsx', index=False)
    
    print(f"\n All cleaned data saved:")
    print(f"  - customers_clean.xlsx")
    print(f"  - vendors_clean.xlsx")
    print(f"  - sales_clean.xlsx")
except Exception as e:
    print(f"\n Could not save files: {e}")


STEP 9: Saving Cleaned Data
--------------------------------------------------------------------------------

 Could not save files: name 'df_customers_clean' is not defined


In [14]:
# =============================================================================
# 10. DATA QUALITY REPORT
# =============================================================================
print("\n" + "=" * 80)
print("STEP 10: Data Quality Report")
print("-" * 80)

def data_quality_report(df, name):
    """Generate a data quality summary"""
    print(f"\n {name} Quality Report")
    print("-" * 40)
    
    total_cells = df.shape[0] * df.shape[1]
    missing_cells = df.isnull().sum().sum()
    completeness = ((total_cells - missing_cells) / total_cells * 100)
    
    print(f"Overall Completeness: {completeness:.2f}%")
    print(f"Total Records: {df.shape[0]:,}")
    print(f"Total Fields: {df.shape[1]}")
    print(f"Missing Cells: {missing_cells:,} / {total_cells:,}")
    
    # Check for potential key columns
    print(f"\nPotential Key Columns:")
    for col in df.columns:
        unique_count = df[col].nunique()
        unique_ratio = unique_count / len(df)
        if unique_ratio > 0.95:
            print(f"  • {col}: {unique_count:,} unique values (likely unique ID)")
        elif unique_ratio < 0.1 and unique_count < 50:
            print(f"  • {col}: {unique_count} categories (likely grouping variable)")

data_quality_report(df_customers_clean, "Customers")
data_quality_report(df_vendors_clean, "Vendors")
data_quality_report(df_sales_clean, "Posted Sales")


STEP 10: Data Quality Report
--------------------------------------------------------------------------------


NameError: name 'df_customers_clean' is not defined

In [17]:
# =============================================================================
# 11. NEXT STEPS & ACTION ITEMS
# =============================================================================
print("\n" + "=" * 80)
print("NEXT STEPS FOR TODAY")
print("=" * 80)
print("""
 All three datasets loaded and cleaned
 Customer list created
  
IMMEDIATE TODO:
□ Review the Posted Sales columns (Step 6) 
□ Identify how sales link to customers (customer_no field?)
□ Look for these critical fields in sales data:
  - Invoice/Document Date
  - Due Date or Payment Date
  - Amount fields
  - Customer identifier
  
ONCE YOU IDENTIFY THE COLUMNS, WE CAN:
□ Calculate days to payment (Due Date - Payment Date)
□ Calculate days overdue
□ Analyze payment patterns by customer
□ Compare actual payment behavior vs payment terms
□ Identify late-paying customers
□ Build payment behavior profiles

THESIS PREP:
□ Start documenting data dictionary
□ Note any data quality issues
□ Identify which customers have complete payment history
□ Consider: What defines "good" vs "bad" payment behavior?

""")

print("=" * 80)
print(f"Analysis completed: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)
print("\n PASTE THE POSTED SALES COLUMNS HERE AND WE'LL BUILD THE PAYMENT ANALYSIS!")
print("=" * 80)


NEXT STEPS FOR TODAY

 All three datasets loaded and cleaned
 Customer list created

IMMEDIATE TODO:
□ Review the Posted Sales columns (Step 6) 
□ Identify how sales link to customers (customer_no field?)
□ Look for these critical fields in sales data:
  - Invoice/Document Date
  - Due Date or Payment Date
  - Amount fields
  - Customer identifier

ONCE YOU IDENTIFY THE COLUMNS, WE CAN:
□ Calculate days to payment (Due Date - Payment Date)
□ Calculate days overdue
□ Analyze payment patterns by customer
□ Compare actual payment behavior vs payment terms
□ Identify late-paying customers
□ Build payment behavior profiles

THESIS PREP:
□ Start documenting data dictionary
□ Note any data quality issues
□ Identify which customers have complete payment history
□ Consider: What defines "good" vs "bad" payment behavior?


Analysis completed: 2026-02-09 14:25:30

 PASTE THE POSTED SALES COLUMNS HERE AND WE'LL BUILD THE PAYMENT ANALYSIS!
